# 03 — Google Colab: analysis, capability gates, human sheet, paper bundle

Run this notebook on a CPU runtime after both generators and the 14B judge have completed. It analyzes the persistent Drive checkpoints directly; there is no Kaggle import/export step.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/political-bias-lab.git"
PROFILE = "smoke"
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
RUN_ID = f"{PROFILE}-{GIT_SHA[:8]}"
RUN_ROOT = f"{DRIVE_ROOT}/runs/{RUN_ID}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-core.txt"])
print("Run ID:", RUN_ID)

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, RUN_ROOT)

In [ ]:
from pathlib import Path
import pandas as pd
raw_dir = Path(REPO_DIR)/"results/raw"
for p in sorted(raw_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    failures = int(df["error"].notna().sum()) if "error" in df.columns else 0
    print(f"{p.name}: rows={len(df):,}, failures={failures:,}")

In [ ]:
from src.config import load_config
from src.pipeline import analyze_all
from pathlib import Path
cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
counts = analyze_all(cfg, root=REPO_DIR)
counts

## Capability gates

These gates protect measurement validity. A failed gate does **not** mean the model is politically biased or unbiased; it means that phase cannot support the intended inference for that model/judge without another methodology revision.

In [ ]:
gates_path = Path(REPO_DIR)/"results/derived/capability_gates.csv"
if gates_path.exists():
    gates = pd.read_csv(gates_path)
    display(gates)
    failed = gates[~gates["passed"].astype(bool)]
    if len(failed):
        print("WARNING: capability gates failed. Do not advance this methodology to a final paper run yet.")
        display(failed)
    else:
        print("All currently available capability gates passed.")

In [ ]:
for name in [
    "phase1_summary.csv", "phase2_capability.csv", "phase2_summary.csv",
    "phase3_overall.csv", "phase3_per_class.csv",
    "phase4_judge_quality.csv", "phase4_judge_summary.csv"
]:
    p = Path(REPO_DIR)/"results/derived"/name
    if p.exists():
        print("\n###", name)
        display(pd.read_csv(p))

## Blinded human evaluation

`phase4_human_rating_sheet.csv` contains randomized Response X/Y pairs and hides generator/condition identity. For the paper, use **at least two independent blinded raters**. Give each rater a copy, combine completed rows into `phase4_human_ratings_completed.csv`, place it in `results/derived/`, and rerun this notebook to calculate inter-rater reliability.

In [ ]:
sheet = Path(REPO_DIR)/"results/derived/phase4_human_rating_sheet.csv"
key = Path(REPO_DIR)/"results/derived/phase4_human_blind_key.parquet"
print("Human rating sheet:", sheet)
print("Keep this key hidden from raters:", key)

## Build evidence bundle

For a final paper run, preserve this bundle unchanged. The bundle includes raw observations, derived statistics, capability gates, figures, and manifests.

In [ ]:
from src.pipeline import build_paper_bundle
import shutil
bundle_dir = build_paper_bundle(cfg, root=REPO_DIR)
zip_path = shutil.make_archive(str(Path(RUN_ROOT)/f"paper_bundle_{PROFILE}_{GIT_SHA[:8]}"), "zip", root_dir=bundle_dir)
print("Paper bundle:", zip_path)

### Pilot → paper rule

After the redesigned pilot passes the capability gates, freeze: Git commit, hypotheses, prompts, model IDs/revisions, real-entity pairs, exclusions, primary metrics, and human-rating protocol. Then run `PROFILE = "paper"` from notebook 00 onward without changing the methodology in response to paper-run results.